# 3D NTL Intensity Map — deck.gl / pydeck

**Author:** Bouchra Daddaoui  
**Repository:** viirs-electrification-ml  

Interactive 3D column map of VIIRS nighttime light radiance using **pydeck** (deck.gl for Python),
inspired by Milos Agathon's 3D population density visualisations.

Each tile is represented as a vertical column whose **height encodes NTL intensity** and whose
**colour encodes energy poverty class** — revealing spatial clusters of electrification inequality in 3D.

**Outputs:**
- `figures/pydeck_brazil_3d.html`
- `figures/pydeck_china_3d.html`
- `figures/pydeck_morocco_3d.html`
- `figures/pydeck_combined_3d.html`

In [ ]:
import sys
sys.path.insert(0, '../scripts')

import numpy as np
import pandas as pd
import geopandas as gpd
import pydeck as pdk
from shapely.geometry import box
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

FIGURES = Path('../figures')
FIGURES.mkdir(exist_ok=True)

print(f'pydeck {pdk.__version__} loaded.')

## 1. Synthetic Tile Data with Elevation Encoding

Tile centroids are used as column positions. NTL radiance → column height (metres).  
Energy poverty class → column colour (blue=electrified, red=energy poor, yellow=unlit).

In [ ]:
def generate_tiles(country, n_tiles, bbox, ntl_mean, ntl_std, seed):
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = bbox
    cols = int(np.sqrt(n_tiles))
    rows = n_tiles // cols
    xs = np.linspace(minx, maxx, cols + 1)
    ys = np.linspace(miny, maxy, rows + 1)
    geoms, tile_ids = [], []
    for i in range(rows):
        for j in range(cols):
            geoms.append(box(xs[j], ys[i], xs[j+1], ys[i+1]))
            tile_ids.append(f'{country[:3].upper()}_{i:02d}_{j:02d}')
    n = len(geoms)
    centroids = np.array([[g.centroid.x, g.centroid.y] for g in geoms])
    dists = np.linalg.norm(centroids[:, None] - centroids[None, :], axis=-1)
    cov = ntl_std**2 * np.exp(-dists / (0.3 * (maxx - minx)))
    ntl = rng.multivariate_normal(np.full(n, ntl_mean), cov)
    ntl = np.clip(ntl, 0, None)
    pop = rng.lognormal(mean=np.log(100), sigma=1.2, size=n)
    gdf = gpd.GeoDataFrame({
        'tile_id': tile_ids, 'country': country,
        'ntl_mean': ntl, 'pop_density': pop,
        'lon': centroids[:, 0], 'lat': centroids[:, 1],
    }, geometry=geoms, crs='EPSG:4326')
    return gdf


def add_energy_class(gdf):
    ntl = gdf['ntl_mean']
    q25, q75 = ntl.quantile(0.25), ntl.quantile(0.75)
    gdf = gdf.copy()
    gdf['energy_class'] = np.where(ntl >= q75, 'Electrified',
                           np.where(ntl >= q25, 'Energy poor', 'Low pop / unlit'))
    # Colours as [R, G, B, A] for pydeck
    colour_map = {
        'Electrified':     [26,  120, 194, 220],
        'Energy poor':     [214,  39,  40, 220],
        'Low pop / unlit': [245, 197,  24, 220],
    }
    gdf['colour'] = gdf['energy_class'].map(colour_map)
    # Elevation scaled: 1 NTL unit → ~2000 m for visual impact
    gdf['elevation'] = gdf['ntl_mean'] * 2000
    return gdf


countries_cfg = [
    dict(country='Brazil',  n_tiles=100, bbox=(-48,-23,-43,-18), ntl_mean=12.5, ntl_std=8.0,  seed=1),
    dict(country='China',   n_tiles=100, bbox=(116,29,122,33),   ntl_mean=28.3, ntl_std=14.0, seed=2),
    dict(country='Morocco', n_tiles=100, bbox=(-17.1,20.8,-1.0,35.9), ntl_mean=8.1, ntl_std=5.5, seed=3),
]

gdfs = {cfg['country']: add_energy_class(generate_tiles(**cfg)) for cfg in countries_cfg}

for c, g in gdfs.items():
    print(f'{c}: {len(g)} tiles | NTL mean={g.ntl_mean.mean():.1f} | elev mean={g.elevation.mean():.0f} m')

## 2. Per-Country 3D Column Maps

Each tile centroid becomes a vertical column. Height = NTL intensity. Colour = energy class.  
Tilt the map (right-click drag) to see the 3D effect.

In [ ]:
# Approximate tile width in metres for disk radius
TILE_RADIUS_M = {
    'Brazil':  30_000,
    'China':   30_000,
    'Morocco': 75_000,   # larger country bbox → bigger tiles
}

PITCH = 45    # degrees tilt
BEARING = -20  # rotation

for country, gdf in gdfs.items():
    df = gdf[['lon', 'lat', 'ntl_mean', 'elevation', 'colour', 'tile_id', 'energy_class']].copy()
    df['colour'] = df['colour'].tolist()  # ensure list format

    bounds = gdf.total_bounds
    centre = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

    layer = pdk.Layer(
        'ColumnLayer',
        data=df,
        get_position=['lon', 'lat'],
        get_elevation='elevation',
        elevation_scale=1,
        radius=TILE_RADIUS_M[country],
        get_fill_color='colour',
        pickable=True,
        auto_highlight=True,
        extruded=True,
    )

    view = pdk.ViewState(
        latitude=centre[0], longitude=centre[1],
        zoom=5, pitch=PITCH, bearing=BEARING,
    )

    tooltip = {
        'html': '<b>{tile_id}</b><br>NTL: {ntl_mean:.1f} nW/cm²/sr<br>Class: {energy_class}',
        'style': {'backgroundColor': '#1a1a2e', 'color': 'white', 'fontSize': '12px'},
    }

    deck = pdk.Deck(
        layers=[layer],
        initial_view_state=view,
        tooltip=tooltip,
        map_style='mapbox://styles/mapbox/dark-v10',
    )

    out = FIGURES / f'pydeck_{country.lower()}_3d.html'
    deck.to_html(str(out))
    print(f'Saved: {out}')

## 3. Combined Multi-Country 3D Map

All three countries in a single deck with individual ColumnLayers.  
World-level view — zoom into any country to explore its electrification landscape in 3D.

In [ ]:
all_layers = []
for country, gdf in gdfs.items():
    df = gdf[['lon', 'lat', 'ntl_mean', 'elevation', 'colour', 'tile_id', 'country', 'energy_class']].copy()
    all_layers.append(
        pdk.Layer(
            'ColumnLayer',
            data=df,
            get_position=['lon', 'lat'],
            get_elevation='elevation',
            elevation_scale=1,
            radius=TILE_RADIUS_M[country],
            get_fill_color='colour',
            pickable=True,
            auto_highlight=True,
            extruded=True,
        )
    )

world_view = pdk.ViewState(latitude=20, longitude=10, zoom=1, pitch=30, bearing=0)

tooltip = {
    'html': '<b>{country} — {tile_id}</b><br>NTL: {ntl_mean:.1f} nW/cm²/sr<br>Class: {energy_class}',
    'style': {'backgroundColor': '#1a1a2e', 'color': 'white', 'fontSize': '12px'},
}

deck_combined = pdk.Deck(
    layers=all_layers,
    initial_view_state=world_view,
    tooltip=tooltip,
    map_style='mapbox://styles/mapbox/dark-v10',
)

out = FIGURES / 'pydeck_combined_3d.html'
deck_combined.to_html(str(out))
print(f'Saved: {out}')
print()
print('Open figures/pydeck_combined_3d.html → right-click drag to tilt into 3D.')

## 4. NTL Distribution by Energy Class

Statistical summary of the NTL distributions that drive the 3D visualisation.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colours = {'Electrified': '#1a78c2', 'Energy poor': '#d62728', 'Low pop / unlit': '#f5c518'}

for ax, (country, gdf) in zip(axes, gdfs.items()):
    for cls, grp in gdf.groupby('energy_class'):
        ax.hist(grp['ntl_mean'], bins=15, alpha=0.7,
                color=colours[cls], label=cls, edgecolor='white', linewidth=0.3)
    ax.set_xlabel('NTL Radiance (nW/cm²/sr)', fontsize=9)
    ax.set_ylabel('Tile count', fontsize=9)
    ax.set_title(country, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('NTL Distribution by Energy Class — drives 3D column height',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES / 'ntl_distribution_by_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/ntl_distribution_by_class.png')